# Semantic feature extraction

**Recipe 35** · Level 3 (Intermediate) · Decision type: `Noul`

Convert text into predefined semantic features with independent judgments and compare a classifier trained on those features with a fixed baseline.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)

## What you will build

A fixed list of six semantic features about one piece of customer feedback: does it mention a price, express frustration, ask for a refund or cancellation, name a competitor, report a technical issue, or express satisfaction. Jev answers all six as independent `Noul` propositions in a single request, giving one feature vector per text. Python then composes that vector with a frozen, per-feature business threshold, trains a small classifier on the vectors to predict whether the feedback should be escalated to a human specialist, and compares it with a fixed baseline on data neither of them ever trained or was tuned on.

The methodological point this recipe teaches matters as much as the pipeline: the six features, the split that each text belongs to, and the baseline are all fixed in code before any `test`-split number is computed. Nothing below is tuned after looking at `test`.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays stored answers offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls and changes nothing else. The header states which mode ran, and every number below carries that mode with it.

The fixture set has 50 texts: 16 `train`, 16 `validation`, 16 `test` and 2 `demo`. `scored` below is `validation` and `test` together (32 texts) — the examples a reported number is ever about; `train` fits the classifier and the baseline, and is never itself scored.

In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_threshold,
    pool_label_decisions,
    select_confidence_threshold,
    select_threshold,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())
# scored excludes train (fits the classifier and the baseline) and demo (shown, never scored):
# see "gold label" under Evaluation for why a demo example carries none.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result, in every run mode, not only an offline one.
selection = " (a selection step, not a reported result)"

run_header(
    35,
    "Semantic feature extraction",
    backend=backend,
    n_examples=len(scored),
)

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl` carries `fields`: a `message_id` and the feedback `text`. `build_state` passes on the text only — the identifier stays in Python and is attached to the outcome at the end, so a message never needs an identifier to pass through the model and every result still traces back to its source.

In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
example = demo["d01"]
print("Python has:")
print(json.dumps(example.fields, indent=2))
state = helpers.build_state(example.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

## The questions

There are six questions, and each asks one thing: does this feedback mention a price, express frustration, ask for a refund or cancellation, name a competitor, report a technical issue, or express satisfaction. Each is a `Noul`, written as a statement that can be true or false, because the six outcomes are not mutually exclusive — a message can do several of these at once, or none of them — so one five-or-six-way `Choice` would be the wrong fit (CONTRIBUTING.md section 3). Python builds all six from one fixed list, `helpers.FEATURES`, so the question wording and the fixtures' own gold labels cannot drift apart, and sends them together in a single request: TypeSafe's primitives page (S02) states that independent questions over the same state belong in one request, and that is exactly what a feature *vector* needs — one request, six independent judgments, one vector per text.

In [ ]:
questions = helpers.build_questions()
for name, question in questions.items():
    print(f"{name}:")
    print(f"  {question.instructions}")

## One answer up close

Before any aggregate, look at one text's six typed answers. A `Noul` answer holds one probability, `noul`: the probability, from 0 to 1, that the statement is true. Unlike a `Choice` or a `Score`, a `Noul` has no separate `confidence` field (S02, S03) — a value near 0.5 is simply the uncertain case. The six answers below are decided together, in the one request `backend.decide` makes for this text, and every text in the fixture set (`train`, `validation`, `test` and `demo` alike) is decided exactly once here and reused everywhere it is shown or scored again, so the request count below is the real number of calls this notebook makes, counted at the one place they happen.

In [ ]:
request_count = 0


def decide_once(example):
    global request_count
    request_count += 1
    return backend.decide(helpers.build_state(example.fields), questions)


answers = {e.id: decide_once(e) for e in examples}
print(
    f"requests made: {request_count} for {len(examples)} texts, "
    f"six features each in one request{check}"
)

for name in helpers.FEATURES:
    print(f"{name}:")
    show_answer(answers["d01"][name])

In [ ]:
probabilities = {name: answers["d01"][name].noul for name in helpers.FEATURES}
for name, p in probabilities.items():
    print(f"{name}: {p:.2f}")
plot_answer_probabilities(probabilities, title="One text's six feature probabilities")

## Python's part

Jev supplies six independent probabilities; everything that turns them into a decision is code, and all of it is fixed before this notebook ever looks at `test`.

**The business threshold, per feature.** A `Noul` has no confidence field (S02), so unlike a recipe that gates a `Choice` or `Score` answer by its own confidence, each feature here gets a plain business threshold: `present = noul >= threshold`, with nothing to defer to a person — a semantic feature is either present or it is not. Each of the six thresholds is chosen separately on `validation` with `jev_cookbook.evaluation.select_threshold` (objective `"f1"`), then frozen. `helpers.compose_features` applies all six comparisons at once and is the deterministic composition this recipe tests directly in `tests/test_helpers.py`: tests/test_helpers.py checks every feature is independently consequential — flipping one feature's own answer never changes another feature's tag, and (more strongly) flipping one feature's own composed bit, on a real committed example, changes what the classifier below predicts.

**The baseline and the classifier, fit on `train` only.** The fixed baseline (`helpers.fit_majority_baseline`) is the majority value of `train`'s own `escalate` [gold label](../../docs/glossary.md#gold-label) — a single constant, chosen once and never updated by a feature vector. The classifier (`helpers.fit_naive_bayes`) is a small Bernoulli Naive Bayes model, plain Python and `math.log`, fit on `train`'s composed feature vectors and its own `escalate` labels, with one never-tuned smoothing constant (`helpers.LAPLACE_ALPHA = 1.0`). Both the baseline and the classifier are frozen in this cell, from `train` alone — before the "Evaluation" section below reads a single `validation` or `test` number.

**A pooled confidence gate, as a diagnostic.** Every published Noul recipe in this cookbook reports a confidence-gated [review](../../docs/glossary.md#review) outcome alongside its business rule, using `jev_cookbook.evaluation.noul_confidence` (S03) to derive a certainty from a Noul's single probability. This recipe's own composition has nothing to defer to a person (there is no third outcome `compose_features` can produce), so the gate below is a diagnostic rather than a routing decision: it is chosen on `validation` exactly the way a three-path recipe chooses one (`select_confidence_threshold`, pooled across all six features with `pool_label_decisions`), frozen here, and reported in "Evaluation" as how many (text, feature) decisions would be sent to [review](../../docs/glossary.md#review) under it — but `helpers.feature_vector` never withholds a feature for low confidence, because the classifier below can weigh a noisy boolean input itself, unlike an automatic answer shown directly to a reader.

In [ ]:
validation = select_split(examples, "validation")
train = select_split(examples, "train")

thresholds = {}
for name in helpers.FEATURES:
    val_gold = [labels[e.id]["features"][name] for e in validation]
    val_noul = [answers[e.id][name].noul for e in validation]
    thresholds[name] = select_threshold(val_gold, val_noul, "f1")
    print(f"{name}: threshold {thresholds[name]:.2f}, frozen from here on{selection}{check}")

In [ ]:
val_gold_by_label = {
    name: [labels[e.id]["features"][name] for e in validation] for name in helpers.FEATURES
}
val_noul_by_label = {
    name: [answers[e.id][name].noul for e in validation] for name in helpers.FEATURES
}
val_correct, val_confidence, _val_keys = pool_label_decisions(
    [e.id for e in validation],
    list(helpers.FEATURES),
    val_gold_by_label,
    val_noul_by_label,
    thresholds,
)
confidence_gate = select_confidence_threshold(val_correct, val_confidence, min_coverage=0.5)
print(f"pooled confidence gate, frozen from here on: {confidence_gate:.2f}{selection}{check}")

In [ ]:
def feature_vector_for(example):
    decisions = helpers.compose_features(answers[example.id], thresholds)
    return helpers.feature_vector(decisions)


train_vectors = [feature_vector_for(e) for e in train]
train_targets = [labels[e.id]["escalate"] for e in train]
baseline = helpers.fit_majority_baseline(train_targets)
model = helpers.fit_naive_bayes(train_vectors, train_targets)
print(
    f"train: {sum(train_targets)} escalate, {len(train_targets) - sum(train_targets)} "
    f"not escalate, of {len(train_targets)}{check}"
)
print(f"baseline, frozen from here on: always predict escalate={baseline.majority}{check}")
print(f"classifier fit on {len(train_vectors)} train feature vectors{check}")

In [ ]:
for example in (demo["d01"], demo["d02"]):
    vector = feature_vector_for(example)
    present = [name for name, flag in zip(helpers.FEATURES, vector, strict=True) if flag]
    print(f"{example.id}: features present = {present or '(none)'}{check}")
    print(
        f"  baseline -> escalate={helpers.predict_majority_baseline(baseline, 1)[0]}, "
        f"classifier -> escalate={helpers.predict_naive_bayes(model, vector)}{check}"
    )

## Evaluation

Every setting above — the six thresholds, the baseline, the classifier — was chosen and frozen on `validation` and `train`. This section reports what the frozen pipeline does, at two levels: **per question**, how well each feature's own business threshold reproduces that feature's own [gold label](../../docs/glossary.md#gold-label) (its [precision](../../docs/glossary.md#precision), [recall](../../docs/glossary.md#recall), [F1](../../docs/glossary.md#f1) and [support](../../docs/glossary.md#support), the number of scored examples whose gold label is true for that feature); and **at task level**, how well the classifier and the baseline predict `escalate` — a second, independently-authored gold label never computed from the six features. `test` is used once, after every choice above was frozen without looking at it.

Two different quantities are at work here, and they should never be compared against each other: there are 32 scored **texts** (16 `validation`, 16 `test`), but each text carries six feature decisions, so the per-question numbers below are counted over 192 **(text, feature) pairs** (96 per split), while the task-level numbers are counted over the 32 texts themselves — one classifier decision per text, not per feature.

In [ ]:
print(
    f"validation, {len(validation)} texts, {len(validation) * len(helpers.FEATURES)} "
    f"(text, feature) pairs{selection}{check}"
)
for name in helpers.FEATURES:
    gold = [labels[e.id]["features"][name] for e in validation]
    noul = [answers[e.id][name].noul for e in validation]
    pt = evaluate_threshold(gold, noul, thresholds[name])
    print(
        f"  {name:35s} P={pt.precision:.4f} R={pt.recall:.4f} F1={pt.f1:.4f} "
        f"support={sum(gold)}/{len(gold)}{selection}{check}"
    )

In [ ]:
test = select_split(examples, "test")
print(
    f"test, {len(test)} texts, {len(test) * len(helpers.FEATURES)} "
    f"(text, feature) pairs, every threshold frozen{check}"
)
for name in helpers.FEATURES:
    gold = [labels[e.id]["features"][name] for e in test]
    noul = [answers[e.id][name].noul for e in test]
    pt = evaluate_threshold(gold, noul, thresholds[name])
    print(
        f"  {name:35s} P={pt.precision:.4f} R={pt.recall:.4f} F1={pt.f1:.4f} "
        f"support={sum(gold)}/{len(gold)}{check}"
    )

### Pooled across all six features

`jev_cookbook.evaluation.pool_label_decisions` flattens every (text, feature) pair of one split into one correctness array, so a single pooled accuracy figure can be read beside the six per-feature ones above — the per-question view, once per split, rather than six times. There is no second, unconditional branch in `helpers.compose_features` (no fallback feature, no deferral): every pair gets a present-or-absent tag from its own threshold alone, so there is nothing for a rule's own outcomes (`jev_cookbook.evaluation.pool_label_outcomes` / `evaluate_outcomes`) to report that this pooled correctness does not already say — that machinery exists for a rule with a review branch beyond a plain threshold comparison, which this recipe's feature rule does not have.

The pooled confidence gate frozen in "Python's part" is reported here too, as the diagnostic it is: how many (text, feature) pairs fall below it, out of how many pairs were pooled. This never changes `helpers.feature_vector`'s own output — the classifier always sees the business tag, confident or not — so a reader can see what gating every feature the way a three-path Noul recipe does would have withheld, without this recipe actually withholding it.

In [ ]:
for split_name, split_examples in (("validation", validation), ("test", test)):
    gold_by_label = {
        name: [labels[e.id]["features"][name] for e in split_examples] for name in helpers.FEATURES
    }
    noul_by_label = {
        name: [answers[e.id][name].noul for e in split_examples] for name in helpers.FEATURES
    }
    correct, confidence, _keys = pool_label_decisions(
        [e.id for e in split_examples],
        list(helpers.FEATURES),
        gold_by_label,
        noul_by_label,
        thresholds,
    )
    label = f"{selection}{check}" if split_name == "validation" else check
    print(
        f"{split_name} pooled accuracy: {sum(correct) / len(correct):.4f} "
        f"({sum(correct)} of {len(correct)} (text, feature) pairs){label}"
    )
    below_gate = sum(1 for c in confidence if c < confidence_gate)
    print(
        f"{split_name}: {below_gate} of {len(confidence)} (text, feature) pairs would be sent "
        f"to review under the pooled confidence gate (diagnostic only, not withheld){label}"
    )

### Task level: the classifier against the fixed baseline

Held-out [accuracy](../../docs/glossary.md#accuracy) of both the classifier and the baseline, over the 32 scored texts (one prediction per text, not per feature). Both were frozen from `train` alone in "Python's part" above; nothing here chooses or adjusts either of them.

In [ ]:
for split_name, split_examples in (("validation", validation), ("test", test)):
    vectors = [feature_vector_for(e) for e in split_examples]
    targets = [labels[e.id]["escalate"] for e in split_examples]
    base_pred = helpers.predict_majority_baseline(baseline, len(split_examples))
    clf_pred = [helpers.predict_naive_bayes(model, v) for v in vectors]
    label = f"{selection}{check}" if split_name == "validation" else check
    print(f"{split_name}, {len(split_examples)} texts{label}")
    print(f"  baseline accuracy:   {accuracy(targets, base_pred):.4f}{label}")
    print(f"  classifier accuracy: {accuracy(targets, clf_pred):.4f}{label}")

In [ ]:
test_vectors = [feature_vector_for(e) for e in test]
test_targets = [labels[e.id]["escalate"] for e in test]
test_clf_pred = [helpers.predict_naive_bayes(model, v) for v in test_vectors]
matrix = confusion_matrix(test_targets, test_clf_pred, labels=[False, True])
print(f"Test confusion matrix, gold rows by predicted columns, classifier only{check}:")
print(f"  {'predicted ->':<16s}  {'False':>5s}  {'True':>5s}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    print(f"  gold {str(gold_label):<11s}  {row[0]:>5d}  {row[1]:>5d}{check}")
plot_confusion_matrix(matrix, title=f"Test split, {len(test)} texts, classifier{check}")

In [ ]:
test_base_pred = helpers.predict_majority_baseline(baseline, len(test))
print("Test texts the classifier got wrong, and what the baseline did on the same text:")
for example, vector, pred, base, gold in zip(
    test, test_vectors, test_clf_pred, test_base_pred, test_targets, strict=True
):
    if pred != gold:
        present = [name for name, flag in zip(helpers.FEATURES, vector, strict=True) if flag]
        print(
            f"  {example.id}: gold escalate={gold}, classifier={pred}, baseline={base}, "
            f"features present={present or '(none)'}"
        )

The cell above is why a fixed boolean feature list is not the whole story: `t12`, a long-time customer quietly leaving with none of the six features present, is missed by the classifier and the baseline alike — there is no feature here for either of them to act on. A classifier built on a feature list that happens to make every test case separable would not be teaching anything, since a comparison that could not possibly come out the other way is not a finding; this one can and does fall short, on both splits, in a way a reader can trace back to specific texts above.

**What the held-out comparison does and does not establish.** `test` is a different 16 texts from `validation`, written separately, never reused: it is "held out" in the sense that neither the six thresholds nor the baseline nor the classifier were chosen by looking at it. It is not independent evidence about escalation decisions in general — 16 hand-written feedback messages, scored by a replay of hand-written probabilities, say only that this pipeline's held-out accuracy for the classifier and for the baseline came out to whatever the task-level cell above printed for `test`. A reported figure that used `test` to pick anything — a threshold, a smoothing constant, which split a text belongs to — would not be a held-out figure at all; this one does not, and that is exactly what "Python's part" freezing everything first is for.

## What was and was not measured

What this run can say about Jev depends on its mode, which the cell below states from `backend.mode`: a synthetic or scripted run is a pipeline check only, on the fixture sample it ran against; a recorded or live run states the model the API returned and when, for exactly the examples it covers. Nothing here says anything about Jev's quality, speed or cost unless that cell reports a recorded or live run.

Three different counts appear below, and none of them is interchangeable with another: 32 scored **texts** (16 `validation`, 16 `test`), 192 scored **(text, feature) pairs** (32 texts times six features — what the per-feature and pooled numbers above are counted over), and 32 **classifier examples** (one escalate decision per text, the same 32 texts as the scored count, but a different quantity: a task-level unit, not a question-level one).

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: {len(validation)} validation and {len(test)} test texts{check}, "
        f"{len(validation) * len(helpers.FEATURES)} and {len(test) * len(helpers.FEATURES)} "
        f"(text, feature) pairs{check}, {len(validation)} and {len(test)} classifier "
        f"examples{check}"
    )
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(
        f"N: {len(validation)} validation and {len(test)} test texts, "
        f"{len(validation) * len(helpers.FEATURES)} and {len(test) * len(helpers.FEATURES)} "
        f"(text, feature) pairs, {len(validation)} and {len(test)} classifier examples"
    )

## Next steps

- Change the smoothing constant `helpers.LAPLACE_ALPHA` and re-run `helpers.fit_naive_bayes` on `train_vectors`/`train_targets` from "Python's part": the classifier's predictions on `test` move, because the fixtures are not construction-perfect for any one setting.
- Add a hard message to `ROWS` in `build_fixtures.py` with the six feature values and the `escalate` label you expect, run `python build_fixtures.py`, and see where the frozen thresholds and the classifier send it.
- Neighbouring recipes: [`02-refund-intent-detection`](../02-refund-intent-detection/) (a single `Noul` with one business threshold, the pattern this recipe applies six times) and [`06-multiple-topic-labels`](../06-multiple-topic-labels/) (several independent `Noul` propositions composed into a tag set, this recipe's starting point), which this recipe builds on and reuses the pattern of, without reusing its fixtures.

A neighbour link above is a folder link, `../NN-slug/`: link only a recipe whose `notebook.ipynb` is already committed on `main`, never one not yet published.